<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_6_Tool_Use_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 6 — Tool Use with Claude
## AI Architect Mastery Program · Hands-On Lab · Beginner-Friendly


---

### 🎬 The scene

You've just joined **TripGenie**, a Bengaluru travel-tech startup building an AI trip assistant for Indian travellers. Your first ticket: *"Users keep asking things like 'is 350 + 200 rupees enough for two train tickets' and 'should I pack an umbrella for Bangalore this weekend' — make the assistant answer both, correctly, every time."*

Plain chat won't cut it. An LLM should never "wing" arithmetic, and it has no live weather feed. What it needs is the ability to **reach for a tool** — the same way you'd reach for a calculator or check a weather app — and know *which* tool to reach for and *when*.

That is exactly what you are going to build today, and by the end you'll understand it well enough to explain it in a system-design interview.

### Who this is for
Beginners, AI engineers-in-training, and anyone who wants to *build* with the Claude API, not just chat with it. No coding background beyond "I can read Python" is assumed — every line is explained.

### What you'll be able to do by the end
- **Explain** what tool use is and why an LLM can't "just know" some things
- **Build** two working custom tools (calculator + weather) and watch Claude choose between them live
- **Implement** the full tool-calling loop with the real Claude API
- **Distinguish** *your* tools (client tools) from Claude's *built-in* tools (web search, code execution, computer use)
- **Try** the equivalent tools inside claude.ai — no code required
- **Discuss** tool use confidently in an interview, with real numbers and real product names

### How each section is structured
| Block | What it gives you |
|---|---|
| 🧠 The idea | Plain-English explanation |
| 💻 Code | A real cell you run against the live Claude API |
| ✅ Checkpoint | A quick question to prove it landed |
| 🎤 Interview angle | How to talk about it out loud |
| 🤯 Fun fact | A verified, real-world detail |



---
## Section 1 — Why Claude Needs Tools At All

### 🧠 The idea

Claude is a **language model**. It's extraordinary at reading, writing, reasoning, and explaining — but on its own it can only produce *text*, from patterns learned during training. It cannot:
- Guarantee a large arithmetic calculation is exactly correct
- Know today's weather, stock price, or exchange rate
- Touch a database, send an email, or click a button

A **tool** (also called a *function*) is how you hand Claude a capability it doesn't have natively. You describe the tool in plain JSON, Claude decides when it's needed, and **your code** — not Claude — actually runs it.

### The toolbox analogy

```
┌──────────────────────────┐
│   Claude's Toolbox       │
├──────────────────────────┤
│ 📐 Calculator Tool       │
│ 🌤️  Weather Tool          │
└──────────────────────────┘
```

Claude reads your question and decides: *Which tool(s), if any, do I need?*

| User Prompt | Tool Used | Why |
|---|---|---|
| "What is 156 + 89?" | Calculator | Pure math — precision matters |
| "What's the weather in Mumbai?" | Weather | Needs live, current data |
| "Calculate 50 × 12 and tell me if it's raining in Delhi" | Both | Two separate needs in one message |
| "Is 2+2 equal to 4?" | None | Claude already knows this cold |
| "What will the weather be for my 345 km trip to Bangalore?" | Weather | The real ask is buried in a longer sentence — Claude still spots it |

> **Accuracy note:** "Claude doesn't know things" is a useful simplification, not the full truth. Claude *does* know an enormous amount from training — including that 2+2=4. Tools exist for the specific gaps: **live/changing data** (today's weather, current prices) and **guaranteed precision** (large or exact calculations), not for everything. An interviewer will want to hear that distinction, not "LLMs know nothing."

### 🤯 Fun fact
Claude is named after **Claude Shannon**, the mathematician who founded information theory in 1948. Fittingly, Anthropic's own documentation uses "When was Claude Shannon born?" as the textbook example for the web search tool you'll meet later in this lab.

### ✅ Checkpoint

**Question:** If you ask Claude "What is the square root of 144?", will it call the calculator tool?

<details>
<summary>Click to reveal the answer</summary>

Most likely **no tool at all** — Claude already knows √144 = 12 from training, the same way you know it without a calculator. It *could* choose to call a calculator tool if one is offered and it wants to double-check, but for a well-known fact like this, a direct answer is normal and correct.

</details>

### 🎤 Interview angle
**Q: "Why can't you just ask the model to be more careful with arithmetic instead of giving it a calculator?"**
**A:** "Because an LLM predicts the next most likely token — it isn't running arithmetic circuits. For a 12-digit multiplication, 'sounding confident' and 'being correct' are not the same thing. A tool call guarantees a deterministic, verifiable result instead of a probabilistic guess."



---
## Section 2 — The Tool-Calling Loop (What Actually Happens)

### 🧠 The idea

```
Step 1: Claude reads your prompt + the tool definitions (JSON schemas)
              ↓
Step 2: Claude decides which tool(s) to use — or none
              ↓
Step 3: Claude returns a "tool_use" block (a REQUEST, not the final answer)
              ↓
Step 4: Your code runs the real function and gets a real result
              ↓
Step 5: You send that result back to Claude as a "tool_result"
              ↓
Step 6: Claude reads the result and writes the final answer
```

### The one sentence that matters most
**Claude never runs your tools.** Claude only ever asks — through a `tool_use` block — for a tool to be run with specific inputs. Your Python code is the one that opens the file, hits the API, or does the math. This is a safety boundary as much as a technical one: nothing happens on your machine, your database, or your bank account unless *your code* decides to act on Claude's request.

### 🤯 Fun fact
Claude can request **more than one tool in the same turn** — this is called *parallel tool use*, and it's on by default in the Claude API. Ask "convert 100 USD to INR and tell me the weather in Chennai" and Claude can fire off both tool requests together instead of one at a time, which is faster and cheaper.

### 🎤 Interview angle
**Q: "Walk me through what stop_reason tells you."**
**A:** "`stop_reason == 'tool_use'` means Claude wants a tool run and is pausing for the result — my code's job is to execute it and send a `tool_result` back. `stop_reason == 'end_turn'` means Claude is done and `content[0].text` is the final answer. Looping on that one flag is the entire tool-use loop."

### ✅ Checkpoint
**True or False:** If Claude asks for the weather tool and your code never calls the weather function, Claude will still somehow know the weather.

<details><summary>Click to reveal</summary>

**False.** Claude only sees what your code sends back in the `tool_result`. If you never run the function, Claude has no way to know the answer — it will either wait, guess badly, or (if built well) tell you it couldn't get the data.

</details>



---
## Section 3 — Building Tool #1: The Calculator

### Anatomy of a tool

Every tool you hand to Claude has **two parts**:

1. **The Schema** — a JSON dictionary that tells Claude the tool's name, what it does, and what inputs it needs. Claude reads *only* this to decide when to use the tool — it never sees your Python code.
2. **The Function** — the real Python that does the work when your code executes the request.

Think of it like a restaurant: the **schema is the menu description** ("Our cookies are chocolate chip, £3"), and the **function is the kitchen** actually baking them.


In [1]:
# Define the calculator tool schema as a Python dictionary
# This is the "menu description" Claude reads to decide when to use it
calculator_schema = {
    "name": "calculator",                      # simple name, no spaces
    "description": "Evaluates a simple math expression like '5 + 3' or '10 * 2'. Use this for any arithmetic that needs to be exact.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A math expression, e.g. '156 + 89' or '350 * 4'",
            }
        },
        "required": ["expression"],
    },
}

def calculator(expression):
    '''The real function: safely evaluates a basic math expression.'''
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return "Error: only basic math symbols are allowed."
    return eval(expression)

# Quick offline check — no API call needed for this one
print(calculator("156 + 89"))


245



### 🎤 Interview angle
**Q: "Why does the schema need a `description`? Isn't the tool name enough?"**
**A:** "Claude only ever sees the schema, never the function body. The `description` is the only signal it has for *when* to reach for this tool versus another one — a vague description causes wrong or missed tool calls, so writing it well is a real engineering skill, not boilerplate."

### ⚠️ Don't mix these up
- ❌ **Wrong:** "The schema runs the calculation."
  ✅ **Right:** The schema only *describes* the tool. Your `calculator()` function does the actual math.
- ❌ **Wrong:** "Claude sees my Python code and decides based on that."
  ✅ **Right:** Claude only ever sees the JSON schema you send it — never your implementation.



---
## Section 4 — Building Tool #2: The Weather Lookup

Same two-part pattern. The only difference: instead of math, we return weather data.

### Why mock data for this lab?
A real weather API needs an API key, has rate limits, and can be slow or flaky over the network. For **learning the tool-calling mechanism**, mock data removes every one of those variables so you can focus on the one thing that matters right now: watching Claude decide when to call a tool. (Later in this lab you'll see how a *production* system would swap this for a live weather API — the schema and the loop don't change at all.)


In [2]:
# Define the weather lookup tool schema
weather_schema = {
    "name": "weather_lookup",
    "description": "Returns current weather conditions for a given city in India.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city name, e.g. 'Mumbai' or 'Bangalore'",
            }
        },
        "required": ["city"],
    },
}

# Mock weather data — swap this dict for a real API call in production
MOCK_WEATHER = {
    "mumbai": {"condition": "Humid, light rain", "temp_c": 29},
    "bangalore": {"condition": "Pleasant, partly cloudy", "temp_c": 23},
    "delhi": {"condition": "Hot and dry", "temp_c": 38},
    "chennai": {"condition": "Hot and humid", "temp_c": 34},
}

def weather_lookup(city):
    '''The real function: looks up mock weather for a city.'''
    data = MOCK_WEATHER.get(city.lower())
    if not data:
        return f"No weather data for {city}."
    return f"{city}: {data['condition']}, {data['temp_c']}°C"

print(weather_lookup("Bangalore"))


Bangalore: Pleasant, partly cloudy, 23°C



### ✅ Checkpoint
**Q1:** Why not just call a real weather API for this lab?
<details><summary>Answer</summary>No API keys, no rate limits, no network flakiness — you learn the tool-calling *mechanism* without fighting infrastructure. The schema shape is identical either way.</details>

**Q2 (scenario):** Your weather tool returns `"No weather data for Kochi."` What should Claude do with that?
<details><summary>Answer</summary>Tell the user honestly it doesn't have weather for Kochi, rather than guessing. A well-designed tool_result should make the "I don't know" path obvious, not silently invite a hallucinated answer.</details>



---
## Section 5 — Wiring It Up: Watch Claude Choose, Live

### The complete loop, in code form

```
Your Prompt
    ↓
Claude reads prompt + sees both tool schemas
    ↓
Claude decides: calculator? weather? both? neither?
    ↓
Claude returns tool_use block(s) with the inputs it wants to use
    ↓
Your code runs the matching Python function(s)
    ↓
Your code sends the result(s) back as tool_result
    ↓
Claude reads the result(s) and writes the final answer
```

Time to connect this to the real API.


In [3]:
# Cell 1 — install the SDK
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 13.3 MB/s eta 0:00:00


In [4]:
# Cell 2 — key + client (Colab Secret named MY_API_KEY)
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"   # fast + cheap, right default for a lab
print("Ready ✅")

Ready ✅


In [5]:
import json

# Both tool schemas, offered to Claude together
tools_list = [calculator_schema, weather_schema]

# Maps a tool name -> the real Python function that runs it
TOOL_FUNCTIONS = {
    "calculator": calculator,
    "weather_lookup": weather_lookup,
}

def run_tool_calling_interaction(prompt, max_turns=5):
    '''The full loop: send prompt, run any requested tools, loop until done.'''
    messages = [{"role": "user", "content": prompt}]

    for turn in range(max_turns):
        reply = client.messages.create(
            model=MODEL,
            max_tokens=500,
            tools=tools_list,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": reply.content})

        if reply.stop_reason != "tool_use":
            final_text = next(b.text for b in reply.content if b.type == "text")
            print("🟢 Claude's final answer:\n", final_text)
            return

        # Claude wants one or more tools run
        tool_results = []
        for block in reply.content:
            if block.type == "tool_use":
                print(f"🔧 Claude requested: {block.name}({block.input})")
                func = TOOL_FUNCTIONS[block.name]
                result = func(**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                })
        messages.append({"role": "user", "content": tool_results})

    print("⚠️ Hit the turn limit — stopping to avoid an infinite loop.")



### Example 1 — Calculator only


In [6]:
prompt_1 = "What is 1500 divided by 30?"
run_tool_calling_interaction(prompt_1)

🔧 Claude requested: calculator({'expression': '1500 / 30'})
🟢 Claude's final answer:
 1500 divided by 30 is **50**.



### Example 2 — Weather only


In [7]:
prompt_2 = "Is it hot in Delhi right now? How does it compare to Bangalore?"
run_tool_calling_interaction(prompt_2)

🔧 Claude requested: weather_lookup({'city': 'Delhi'})
🔧 Claude requested: weather_lookup({'city': 'Bangalore'})
🟢 Claude's final answer:
 Yes, it's definitely hot in Delhi right now at **38°C** with hot and dry conditions.

In comparison, **Bangalore is much cooler** at **23°C** with pleasant, partly cloudy weather. 

The temperature difference is significant - Delhi is **15°C hotter** than Bangalore. If you're planning travel to either city, you'll want to pack light, breathable clothing for Delhi and perhaps a light jacket or sweater for the more pleasant conditions in Bangalore.



### Example 3 — Both tools, one message (the TripGenie moment)


In [8]:
prompt_3 = (
    "Calculate 350 + 200 rupees for two train tickets. "
    "Also tell me the weather in Bangalore. "
    "If the weather is good, would it be a nice day to travel there from Mumbai?"
)
run_tool_calling_interaction(prompt_3)

🔧 Claude requested: calculator({'expression': '350 + 200'})
🔧 Claude requested: weather_lookup({'city': 'Bangalore'})
🟢 Claude's final answer:
 Great! Here are the results:

**Train Ticket Cost:** 350 + 200 = **₹550** for two train tickets.

**Bangalore Weather:** The weather in Bangalore is **pleasant with partly cloudy conditions and a temperature of 23°C**.

**Travel Recommendation:** Yes, it would be a nice day to travel from Mumbai to Bangalore! The weather conditions are quite favorable - 23°C is a comfortable temperature (neither too hot nor too cold), and partly cloudy skies suggest you'll have some cloud cover without the intensity of direct sunshine. These are ideal conditions for travel. The pleasant weather should make your journey comfortable!



### 🎉 Try this
Change `prompt_3` to something *you* come up with — mix math and weather in one sentence, or ask something that needs neither tool, and watch Claude decide correctly each time.

### 🎤 Interview angle
**Q: "How would you stop this loop running forever?"**
**A:** "A turn limit, like `max_turns=5` above. In production you'd also log every tool call for observability, and consider a timeout per tool so one slow function doesn't stall the whole conversation."



---
## Section 6 — Claude's Built-In Tools (You Don't Have to Build Everything)

### 🧠 The idea

Calculator and weather were **custom tools** — you wrote the schema *and* the function, and your own code ran them (these are called **client tools**). Anthropic also ships a set of **built-in tools** — sometimes called **server tools** — where Anthropic's own infrastructure runs the tool for you. You just add one line to your `tools` list; there's no function of your own to write.

| Built-in tool | What it does | Runs on |
|---|---|---|
| `web_search` | Searches the live web and returns cited sources | Anthropic's servers |
| `web_fetch` | Fetches and reads a specific URL | Anthropic's servers |
| `code_execution` | Runs Python/Bash in a sandbox Anthropic hosts | Anthropic's servers |
| `bash` | Requests a shell command — **your** app runs it and returns the output | Your infrastructure (client tool) |
| `text_editor` | Requests a file edit — pairs with `bash` for agentic coding | Your infrastructure (client tool) |
| `computer_use` | Takes screenshots and controls a mouse/keyboard in a desktop environment | Your infrastructure (client tool) |
| `memory` | Lets Claude write and read notes to persist context across sessions | Your infrastructure (client tool) |

> **Accuracy note:** not all "built-in" tools are server-run. `web_search`, `web_fetch`, and `code_execution` genuinely execute on Anthropic's infrastructure — you just see the results. `bash`, `text_editor`, `computer_use`, and `memory` are **client tools too**: Claude requests them, but *your* application executes the command, edit, click, or memory write, exactly like the calculator tool above. The one-line simplicity is about the schema being pre-built for you, not about who runs the code.

### TripGenie, upgraded
Back in Section 1, you built a custom `weather_lookup` with mock data. A real TripGenie couldn't ship with fake weather — but it also shouldn't reinvent a weather-scraping tool. This is exactly where `web_search` earns its place: real, current, cited information, with zero custom function code.

### 🤯 Fun fact
Web search on the Claude API costs **$10 per 1,000 searches**, on top of normal token cost — and every search result comes back with a citation (URL, title, and the exact snippet used), because Anthropic requires that any web-sourced content shown to end users can be traced back to its source.


In [9]:
# A real call using Claude's BUILT-IN web search tool — no custom function needed
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Search the web: what is today's USD to INR exchange rate?"}],
    tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 3}],
)

# Print just the final text Claude wrote after reading the search results
for block in response.content:
    if block.type == "text":
        print(block.text)


Based on the latest data, 
the current USD/INR exchange rate is 95.330
. 

This means 1 US Dollar equals approximately 95.33 Indian Rupees as of today (July 11, 2026). 
Today's USD/INR range is from 95.222 to 95.419
.

Note that exchange rates fluctuate constantly throughout the day, so the actual rate you receive may vary slightly depending on your bank or money transfer service and when you conduct the transaction.



### What just happened, step by step
1. Claude read your prompt and decided it needed current information → it wrote a search query itself.
2. **Anthropic's servers** ran the actual search (you never wrote a search function).
3. The results came back into the same response, complete with citations.
4. Claude read the results and wrote a normal text answer, same as any other turn.

Compare this to your custom `weather_lookup`: same *shape* of loop, but for `web_search`, Anthropic runs Step 2 for you.

### ⚠️ Don't mix these up
- ❌ **Wrong:** "Built-in tools mean Claude runs code on its own, unsupervised."
  ✅ **Right:** For server tools like `web_search`, Anthropic's infrastructure executes the request within the bounds *you* set (`max_uses`, `allowed_domains`, etc.) — you're still the one turning the tool on and configuring its limits.
- ❌ **Wrong:** "`bash` and `computer_use` mean Claude can control my computer right now, no code needed."
  ✅ **Right:** These are client tools — Claude only *requests* a command or click. Your application (or a product like Claude in Chrome / Claude Code) has to be the one that actually executes it, with whatever safety checks you build in.

### 🎤 Interview angle
**Q: "When would you build a custom tool instead of using `web_search`?"**
**A:** "When the data isn't on the public web — internal databases, proprietary APIs, a company's own inventory system — or when I need guaranteed structure and low latency that a general web search can't promise. `web_search` is for open, current, public information; custom tools are for *your* systems."

### 🤯 Fun fact
Anthropic's **computer use** tool, launched October 22, 2024 alongside Claude 3.5 Sonnet, was the first time a frontier AI model could see a screenshot, decide where to move a cursor, and click — the same way a person operating a computer would. It's the foundation behind products like Claude in Chrome today.

### 🤯 Fun fact
The **Model Context Protocol (MCP)** — the open standard that lets Claude connect to outside tools like Slack, GitHub, or a database — was open-sourced by Anthropic on **November 25, 2024**. It was later adopted by OpenAI and Google DeepMind too, which is unusually fast, cross-company agreement for an AI infrastructure standard.



---
## Section 7 — The Same Ideas, Inside claude.ai (No Code Needed)

### 🧠 The idea

Everything above was the **API** — what you build as a developer. But the person using **claude.ai** in a browser gets many of the same tool-use capabilities as pre-built, one-click features. Recognizing the connection between "the API concept" and "the product feature" is exactly what makes you dangerous in a client demo.

| claude.ai feature | Which concept from today it maps to | Try it yourself |
|---|---|---|
| **Web search toggle** | The `web_search` built-in tool, wrapped in a UI switch | Ask "what happened in the news today" — Claude decides whether to search, same as your API call in Section 6 |
| **Artifacts** | A live code/document panel next to the chat | Ask Claude to "build a small tip calculator" — it writes and renders real code you can interact with |
| **Analysis tool** | A sandboxed JavaScript code-execution environment (similar spirit to `code_execution`) | Upload a CSV and ask Claude to "find the average" — it writes and runs code to compute the real answer, not a guess |
| **Google Workspace / Drive / Calendar connectors** | Custom tools, pre-built for you — same schema-and-function idea as your `calculator`, just pointed at Google's APIs | Connect Google Drive and ask Claude to summarize a real doc |
| **MCP connectors (Slack, GitHub, Notion, 200+ others)** | Custom tools built by other companies, plugged in via the open MCP standard | Connect Slack and ask "summarize yesterday's #general channel" |

### 🤯 Fun fact
As of 2026, **over 200 official and community MCP servers** exist — meaning the "toolbox" pattern you built by hand today (schema + function) has been repeated by hundreds of companies to connect Claude to their own products.

### ✅ Checkpoint
**Scenario:** A colleague says "claude.ai doesn't use tool calling, that's only an API thing." Are they right?

<details><summary>Answer</summary>

**No.** claude.ai is a *product built on top of* the same Claude API and the same tool-use mechanism you used today. The web search toggle, Artifacts, the Analysis tool, and every connector are tools — just pre-built and wrapped in a friendly UI so a non-developer never has to write a JSON schema.

</details>

### 🎤 Interview angle
**Q: "A client wants a demo of 'AI that can look things up.' What do you show them?"**
**A:** "Two options depending on their team: if they want a no-code proof of concept, I'd flip on Web Search in claude.ai live in the room. If they're evaluating for their own product, I'd show the exact `web_search` API call from Section 6 — same underlying capability, but now it's something their engineers can wire into their own app."



---
## Section 8 — How TripGenie Fits Together (Architecture)

```
                    ┌─────────────────────┐
   User message  →  │   Your Application   │
                    │  (holds the messages │
                    │   list + tool funcs)  │
                    └──────────┬───────────┘
                               │  messages.create(tools=[...])
                               ▼
                    ┌─────────────────────┐
                    │     Claude API       │
                    │  decides which tool  │
                    │  (or none) to call   │
                    └──────────┬───────────┘
                 tool_use for          tool_use for
                 CLIENT tool           SERVER tool
                 (calculator,          (web_search runs
                  weather_lookup)       on Anthropic infra)
                       │                      │
                       ▼                      ▼
            ┌────────────────────┐   ┌─────────────────────┐
            │  Your function runs │   │  Anthropic runs the  │
            │  (your code, your   │   │  search, returns     │
            │  infra, your keys)  │   │  cited results        │
            └──────────┬──────────┘   └──────────┬───────────┘
                        └───────────┬─────────────┘
                                    ▼
                        tool_result sent back to Claude
                                    ▼
                        Claude writes the final answer
                                    ▼
                              Back to the user
```

### Component responsibilities
- **Your application**: owns the `messages` list, decides the turn limit, executes client tools, and is the *only* place a real-world action (spend money, send an email, click a button) should be gated behind human review for anything risky.
- **Claude API**: decides *whether* and *which* tool to call, and writes the final natural-language answer.
- **Server tools (web_search, code_execution)**: run on Anthropic's infrastructure; you only configure limits (`max_uses`, `allowed_domains`).

### Failure points & production tradeoffs
- **Bad schema description** → Claude picks the wrong tool, or no tool at all. Fix: write descriptions the way you'd brief a new teammate, with examples.
- **No turn limit** → an infinite loop if a tool result keeps triggering more tool calls. Fix: always cap `max_turns`.
- **Trusting tool_result blindly** → if `weather_lookup` fails silently, Claude may still sound confident. Fix: make errors explicit strings like `"Error: no data for Kochi"`, not empty results.
- **Cost** → server tools like `web_search` bill per call ($10/1,000 searches) *plus* tokens for the search content that lands in context. Fix: set `max_uses` deliberately.
- **Security** → a `bash` or `computer_use` client tool can, in principle, be asked to do something destructive. Fix: never auto-execute a risky action (refunds, deletes, payments) — add a human "type yes to confirm" step in your own code before running it.



---
## Mini Project — Ship "TripGenie Lite"

**Business use case:** TripGenie wants a v1 assistant that can do simple trip-cost math, tell users the weather, *and* now — using what you learned in Section 6 — look up something real from the web (e.g. today's train fares or a live currency rate) without you writing a scraper.

### Build steps
1. Keep `calculator_schema` and `weather_schema` exactly as built above.
2. Add the built-in `web_search` tool to the same `tools_list` your loop already uses — note `run_tool_calling_interaction` needs a small update, because `web_search` doesn't have a matching entry in `TOOL_FUNCTIONS` (Anthropic runs it, not you).
3. Try a prompt that could plausibly need all three: *"Calculate 4500 / 3 for the group's train fare split, tell me the weather in Mumbai, and search for today's INR to USD rate."*
4. Watch the `🔧 Claude requested:` print statements — notice `web_search` never appears there in the same way, because it's a server tool your loop doesn't manually execute.

### Definition of done
- All three tools are declared in one `tools` list.
- A single prompt using all three completes without your loop crashing on the unrecognized `web_search` "tool".
- You can explain, out loud, why `web_search` doesn't need an entry in `TOOL_FUNCTIONS`.

### 🎯 Stretch goal
Swap `weather_lookup`'s mock dictionary for a real weather API call (e.g. Open-Meteo, which needs no key) — the schema doesn't change at all, only the function body. That's the whole point of the schema/function split.


In [10]:
# Starter for the mini project — try updating run_tool_calling_interaction
# to skip execution when block.name == "web_search" (Anthropic already ran it).

tools_with_search = [
    calculator_schema,
    weather_schema,
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 2},
]

print("Tools ready:", [t.get("name") for t in tools_with_search])


Tools ready: ['calculator', 'weather_lookup', 'web_search']



---
## Session Summary

Today you went from "Claude is just a chatbot" to "Claude is an engine that can request real actions, which your code (or Anthropic's) carries out." You built two custom tools by hand, ran the full request → tool_use → tool_result → answer loop against the live API, then saw the exact same pattern power Anthropic's own built-in tools (`web_search`, `code_execution`, `computer_use`, `bash`) and the everyday features inside claude.ai (Web Search toggle, Artifacts, Analysis tool, connectors).

## What You Learned Today
- ✅ You can **explain** why an LLM needs tools for precision and live data, not "everything"
- ✅ You can **build** a schema + function pair from scratch
- ✅ You can **implement** the full tool-calling loop (`stop_reason == "tool_use"` → run → send back → repeat)
- ✅ You can **distinguish** client tools (you run it) from server tools (Anthropic runs it)
- ✅ You can **map** claude.ai features back to the API concepts underneath them
- ✅ You can **discuss** real numbers (web search pricing, MCP launch date, computer use launch date) in an interview

## AI Architect Cheat Sheet
| Concept | One-line definition |
|---|---|
| Tool | A schema + function pair that gives Claude a capability it lacks natively |
| Client tool | Claude requests it; **your app** executes it (calculator, weather, bash, computer_use) |
| Server tool | Claude requests it; **Anthropic's infra** executes it (web_search, web_fetch, code_execution) |
| `tool_use` | Claude's request block — the name + inputs it wants to call |
| `tool_result` | What you send back with the real output |
| `stop_reason` | `"tool_use"` = wants a tool run; `"end_turn"` = final answer is ready |
| Parallel tool use | Claude can request multiple tools in one turn (on by default) |
| MCP | Open standard (Nov 2024) for plugging external tools into Claude |

## 5-Minute Revision Guide
1. A tool = schema (what Claude reads) + function (what your code runs).
2. Claude never runs code — it only ever asks, via `tool_use`.
3. Loop: send prompt → check `stop_reason` → if `tool_use`, run it and send `tool_result` → repeat until `end_turn`.
4. Client tools = you run them. Server tools = Anthropic runs them. Same JSON pattern either way.
5. claude.ai's Web Search toggle, Artifacts, Analysis tool, and connectors are all the same tool-use mechanism, pre-built into a UI.

## Interview Preparation Notes
- **Beginner:** "What is a tool in the Claude API?" → schema + function pair; Claude decides when to call it, your code executes it.
- **Intermediate:** "How do you send a tool's result back to Claude?" → append a `tool_result` block (with the matching `tool_use_id`) to the messages list, as a `user` message, then call `messages.create` again.
- **Advanced:** "What happens if two tools are requested in the same turn?" → parallel tool use is default-on; you loop over every `tool_use` block in `reply.content`, run each, and return all `tool_result`s together in one follow-up message.
- **Architecture:** "Design a travel assistant that needs live pricing, live weather, and internal booking data." → live pricing/weather via server tools (`web_search`) or a client weather API tool; internal booking data via a **custom client tool** hitting your own database, gated by auth; a turn limit and logging for observability.
- **FDE:** "A client is worried an AI agent might book something without permission." → distinguish tools that only *read* data (safe to automate) from tools that *act* (require a human confirmation step in your own code before executing) — this is a design choice you make in the client tool's function, not something the API enforces for you.

## Assignment
- **Beginner:** Add a third custom tool, `currency_converter(amount, from_currency, to_currency)`, using a fixed mock exchange-rate dictionary.
- **Intermediate:** Modify `run_tool_calling_interaction` so it prints how many total tool calls happened across the whole conversation.
- **Advanced:** Add the built-in `web_search` tool to the loop and handle the case where a `tool_use` block's name isn't in `TOOL_FUNCTIONS` (i.e. skip client-side execution for server tools).
- **Project:** Sketch (in Markdown, no need to fully build) a "Support Copilot" for a company that needs: an internal order-lookup tool (client, custom), a `web_search` tool for public policy questions, and a human-confirmation step before any refund tool runs.

## Assessment

**Multiple Choice (10)**
1. What does Claude return when it wants to use a tool? *(a) the final answer (b) a `tool_use` block (c) an error (d) nothing)* → **(b)**
2. Who executes a **client tool**? *(a) Claude (b) Anthropic's servers (c) your application (d) the user manually)* → **(c)**
3. Who executes the built-in `web_search` **server tool**? *(a) your application (b) Anthropic's infrastructure (c) the browser (d) nobody, it's simulated)* → **(b)**
4. What does `stop_reason == "end_turn"` mean? *(a) an error occurred (b) Claude wants a tool run (c) Claude is done and has a final answer (d) the conversation was reset)* → **(c)**
5. Is parallel tool use on by default in the Claude API? *(a) Yes (b) No (c) Only for Opus (d) Only in claude.ai)* → **(a)**
6. What field connects a `tool_result` to the `tool_use` it answers? *(a) `name` (b) `tool_use_id` (c) `role` (d) `index`)* → **(b)**
7. Which of these is a **server** tool? *(a) calculator (b) weather_lookup (c) web_search (d) a custom database lookup)* → **(c)**
8. What does the web search tool return alongside each result? *(a) nothing extra (b) a citation with URL and title (c) a screenshot (d) a video)* → **(b)**
9. When was MCP open-sourced by Anthropic? *(a) Jan 2024 (b) Nov 2024 (c) Jun 2025 (d) It hasn't been)* → **(b)**
10. What is the risk of not setting a turn limit in your tool-calling loop? *(a) higher cost only (b) Claude refuses to answer (c) a potential infinite loop (d) nothing, it self-limits)* → **(c)**

**Short Answer (5)**
1. In one sentence, what is the difference between a tool's schema and its function?
2. Why might Claude choose *not* to use a calculator tool for "2+2"?
3. Name one client tool and one server tool from today's lab.
4. What should a well-designed tool_result do when the underlying lookup fails?
5. Why does a shared schema make it easy to swap mock weather data for a real API later?

**Scenario (3)**
1. Your weather tool starts silently returning `None` instead of an error string when a city isn't found. What's the risk, and how would you fix it?
2. A client wants an agent that can send refund emails automatically. What would you push back on, and what would you suggest instead?
3. Your `run_tool_calling_interaction` loop hits `max_turns` on a real user prompt. What are two possible root causes you'd check first?

## Answer Key (Short Answer & Scenario — model answers)
- **SA1:** The schema is the JSON description Claude reads to decide *when* to call a tool; the function is the real code that runs *what* the tool actually does.
- **SA2:** Claude already knows 2+2=4 confidently from training — tools exist for precision-critical or live/changing data, not everything.
- **SA3:** Client tool: `calculator` (or `weather_lookup`, `bash`, `computer_use`). Server tool: `web_search` (or `web_fetch`, `code_execution`).
- **SA4:** It should return an explicit, human-readable error string (e.g. `"No weather data for Kochi"`) rather than an empty or null value, so Claude can honestly tell the user instead of guessing.
- **SA5:** Because the *schema* Claude reads never changes — only the function body does — swapping mock data for a real API call requires no change to how Claude decides to use the tool.
- **Scenario 1:** Risk — Claude may treat `None` as "no issue" and hallucinate a plausible-sounding weather answer instead of admitting it doesn't know. Fix — always return an explicit error string that Claude can read and relay honestly.
- **Scenario 2:** Push back on fully automatic refunds — a client tool that moves money should never be "fire and forget." Suggest requiring a human confirmation step (in your own application code) before the refund tool actually executes, even though Claude can request it.
- **Scenario 3:** Check whether (a) a tool's function is returning a result that triggers Claude to ask for *another* tool call in a loop (e.g. an ambiguous or malformed result), or (b) the prompt genuinely needs more than `max_turns` steps and the limit is simply too low for this use case.



---
## Additional Resources

### Official documentation
- **Tool use overview**: https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview
- **Web search tool**: https://platform.claude.com/docs/en/agents-and-tools/tool-use/web-search-tool
- **Server tools**: https://platform.claude.com/docs/en/agents-and-tools/tool-use/server-tools
- **Computer use tool**: https://platform.claude.com/docs/en/agents-and-tools/tool-use/computer-use-tool
- **Model Context Protocol**: https://www.anthropic.com/news/model-context-protocol
- **Messages API reference**: https://platform.claude.com/docs/en/api/messages
- **Python SDK**: https://github.com/anthropics/anthropic-sdk-python

### Try in claude.ai
- Web Search toggle, Artifacts, the Analysis tool, and Google Workspace / MCP connectors are all in the tools/settings menu of your claude.ai chat.
